# Pipeline SHAP — GFP Implementation Gap

**Descripción:** Calcula e interpreta los SHAP values del modelo XGBoost óptimo para cada modalidad.
Requiere haber corrido primero `02_modeling_pipeline.ipynb` (modelos `.joblib` ya entrenados).

**Flujo:**
1. Configuración de modalidad y modelo
2. Carga de data procesada + split train/test (idéntico al de modelado)
3. Carga del modelo XGB entrenado
4. Cálculo de SHAP values (TreeExplainer)
5. Tabla SHAP → Excel
6. Summary plot (beeswarm) → PNG

## 0. Configuración — cambiar aquí para cada modalidad

In [ ]:
# ============================================================
# CONFIGURACIÓN — preconfigurado, no necesita cambios
# ============================================================

MODALIDAD    = 'arcc'
SAMPLING     = 'o'
TOP_N        = 10
RANDOM_STATE = 2023
TEST_SIZE    = 0.2

PATH_DATA  = f'C:/15_GFP/data/processed/{MODALIDAD}/1_data_{MODALIDAD}.xlsx'
PATH_MODEL = f'C:/15_GFP/outputs/models/{MODALIDAD}/xgb_{SAMPLING}.joblib'
DIR_SHAP   = f'C:/15_GFP/outputs/shap/{MODALIDAD}'

print(f'Modalidad: {MODALIDAD.upper()} | Sampling: {SAMPLING.upper()} | Top N: {TOP_N}')
print(f'Modelo: {PATH_MODEL}')


## 1. Importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print('Librerías cargadas OK')


## 2. Carga de data y split train/test

In [ ]:
data = pd.read_excel(PATH_DATA, engine='openpyxl')

dep_var = 'brecha_existente'

# Alinear features con las que vio el modelo (puede haber cols extra en data procesada)
model_tmp = joblib.load(PATH_MODEL)
model_features = model_tmp.get_booster().feature_names
del model_tmp

X = data[model_features]
y = data[dep_var]

# Split idéntico al de 02_modeling_pipeline.ipynb
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y,
)

print(f'Train: {x_train.shape[0]:,} | Test: {x_test.shape[0]:,} | Features: {x_train.shape[1]}')
print(f'Distribución y_train: {dict(y_train.value_counts())}')


## 3. Carga del modelo XGB entrenado

In [ ]:
model = joblib.load(PATH_MODEL)
print(f'Modelo cargado: xgb_{SAMPLING} ({MODALIDAD.upper()})')
print(f'Tipo: {type(model).__name__} | Features: {len(model.get_booster().feature_names)}')


## 4. SHAP values (TreeExplainer)

In [ ]:
# XGBoost native pred_contribs — equivalente a SHAP TreeExplainer
# Evita crash de shap 0.51 con Python 3.14 (heap corruption en C++)
SHAP_SAMPLE = 2000

x_shap = x_train.sample(n=min(SHAP_SAMPLE, len(x_train)), random_state=RANDOM_STATE)

dmat      = xgb.DMatrix(x_shap, feature_names=model_features)
contribs  = model.get_booster().predict(dmat, pred_contribs=True)
shap_values = contribs[:, :-1]  # última col = expected value (bias), se excluye

print(f'SHAP values shape: {shap_values.shape}')  # (n_sample, n_features)


## 5. Tabla SHAP → Excel

In [ ]:
shap_mean_abs = np.abs(shap_values).mean(axis=0)

shap_df = (
    pd.DataFrame({'feature': model_features, 'shap_mean_abs': shap_mean_abs})
    .sort_values('shap_mean_abs', ascending=False)
    .reset_index(drop=True)
)

os.makedirs(DIR_SHAP, exist_ok=True)
out_xlsx = f'{DIR_SHAP}/shap_values_xgb_{SAMPLING}.xlsx'
shap_df.to_excel(out_xlsx, index=False)
print(f'Tabla SHAP guardada: {out_xlsx}')

shap_df.head(TOP_N)


## 6. Summary plot (beeswarm) — Top N features

In [ ]:
import shap

top_features = shap_df['feature'].head(TOP_N).tolist()
top_idx      = [list(model_features).index(f) for f in top_features]

def shorten_label(name, max_len=30):
    """Acorta nombres de variables para los gráficos SHAP."""
    # tipo_obra_full_A_B_C → toma el último segmento no genérico
    if name.startswith('tipo_obra_full_'):
        parts = name.replace('tipo_obra_full_', '').split('_')
        # Tomar último segmento; si es 'Otra Infraestructura', usar el anterior
        label = parts[-1].strip()
        if label.lower() in ('otra infraestructura', 'otra') and len(parts) >= 2:
            label = parts[-2].strip()
        return ('tf: ' + label)[:max_len]
    # tipo_obra_nivelN_X → 'tN: X'
    if name.startswith('tipo_obra_nivel'):
        rest = name[len('tipo_obra_nivel'):]
        n, val = rest[0], rest[2:]
        return (f't{n}: ' + val)[:max_len]
    # naturaleza_obra_X → 'nat: X'
    if name.startswith('naturaleza_obra_'):
        return ('nat: ' + name[len('naturaleza_obra_'):])[:max_len]
    # Region_X → 'reg: X'
    if name.startswith('Region_'):
        return ('reg: ' + name[len('Region_'):])[:max_len]
    # modalidad_ejecucion_X → 'mod: X'
    if name.startswith('modalidad_ejecucion_'):
        return ('mod: ' + name[len('modalidad_ejecucion_'):])[:max_len]
    # Otros: reemplazar _ por espacio y truncar
    return name.replace('_', ' ')[:max_len]

display_names = [shorten_label(f) for f in top_features]

shap.summary_plot(
    shap_values[:, top_idx],
    x_shap[top_features].rename(columns=dict(zip(top_features, display_names))),
    feature_names=display_names,
    plot_size=(12, max(6, TOP_N * 0.4)),
    show=False,
    color_bar=True,
)
plt.title(f'SHAP Beeswarm — XGB {SAMPLING.upper()} ({MODALIDAD.upper()}) — Top {TOP_N}')
plt.tight_layout()
out_beeswarm = f'{DIR_SHAP}/shap_beeswarm_xgb_{SAMPLING}.png'
plt.savefig(out_beeswarm, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f'Beeswarm guardado: {out_beeswarm}')


## 7. [Opcional] Bar plot — importancia media absoluta

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(5, TOP_N * 0.35)))
plot_df = shap_df.head(TOP_N).copy().iloc[::-1]
plot_df['label'] = plot_df['feature'].apply(shorten_label)
ax.barh(plot_df['label'], plot_df['shap_mean_abs'], color='steelblue')
ax.set_xlabel('SHAP mean |value|')
ax.set_title(f'Feature Importance SHAP — XGB {SAMPLING.upper()} ({MODALIDAD.upper()}) — Top {TOP_N}')
plt.tight_layout()
out_bar = f'{DIR_SHAP}/shap_bar_xgb_{SAMPLING}.png'
plt.savefig(out_bar, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f'Bar plot guardado: {out_bar}')
